# Product Image Enhancer - Colab Notebook

This notebook is a standalone version of the image enhancement component. It lets you upload an inventory image, choose either `Real-ESRGAN x4plus` or `pillow_fallback`, enhance the image, compare before/after, and export an API-style base64 response.

**Recommended Colab runtime:** `Runtime > Change runtime type > T4 GPU`.

CPU works, but Real-ESRGAN can be slow on larger images.

In [ ]:
# @title 1. Install dependencies
import importlib.util
import subprocess
import sys

def pip_install(*packages):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])

if importlib.util.find_spec("torch") is None or importlib.util.find_spec("torchvision") is None:
    pip_install("torch", "torchvision")

# BasicSR/Real-ESRGAN need a compact inference stack. We install their heavy training/face extras separately only if needed.
pip_install(
    "numpy<2.0",
    "pillow",
    "opencv-python-headless==4.10.0.84",
    "scipy==1.13.1",
    "scikit-image==0.24.0",
    "lmdb==2.2.0",
    "addict==2.4.0",
    "future==1.0.0",
    "requests==2.32.5",
    "tb-nightly==2.21.0a20251023",
    "tqdm==4.67.3",
    "yapf==0.43.0",
    "wheel==0.46.3",
    "Cython==3.2.4",
)
pip_install("--no-deps", "basicsr==1.4.2", "realesrgan==0.3.0")

print("Dependencies installed.")

In [ ]:
# @title 2. Download Real-ESRGAN weights
from pathlib import Path
import urllib.request

WEIGHTS_PATH = Path("/content/RealESRGAN_x4plus.pth")
WEIGHTS_URL = "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth"

if not WEIGHTS_PATH.exists():
    print("Downloading RealESRGAN_x4plus weights. This is about 64 MB.")
    urllib.request.urlretrieve(WEIGHTS_URL, WEIGHTS_PATH)

print(f"Weights ready: {WEIGHTS_PATH} ({WEIGHTS_PATH.stat().st_size / (1024 * 1024):.1f} MB)")

In [ ]:
# @title 3. Define the enhancement pipeline
from __future__ import annotations

import base64
import io
import sys
import types
from dataclasses import dataclass
from typing import Literal

import numpy as np
import torch
import torchvision.transforms.functional as TVF
from PIL import Image, ImageDraw, ImageEnhance, ImageFilter, ImageOps, UnidentifiedImageError

# BasicSR 1.4.2 imports torchvision.transforms.functional_tensor, which was removed in newer torchvision.
# This shim keeps the notebook working on current Colab runtimes.
if "torchvision.transforms.functional_tensor" not in sys.modules:
    functional_tensor = types.ModuleType("torchvision.transforms.functional_tensor")
    functional_tensor.rgb_to_grayscale = TVF.rgb_to_grayscale
    sys.modules["torchvision.transforms.functional_tensor"] = functional_tensor

from basicsr.archs.rrdbnet_arch import RRDBNet
from realesrgan import RealESRGANer

PresetName = Literal["product_standard", "product_detail", "product_soft"]
EngineName = Literal["realesrgan", "pillow_fallback"]

@dataclass(frozen=True)
class EnhancementResult:
    image_bytes: bytes
    width: int
    height: int
    engine: EngineName
    engine_label: str
    preset: PresetName

class ProductImageEnhancer:
    def __init__(self, weights_path: Path, max_input_pixels: int = 24_000_000, target_long_edge: int = 1800):
        self.weights_path = weights_path
        self.max_input_pixels = max_input_pixels
        self.target_long_edge = target_long_edge
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self._realesrgan = None

    def enhance(self, file_bytes: bytes, preset: PresetName = "product_standard", engine: EngineName = "pillow_fallback") -> EnhancementResult:
        image = self._decode(file_bytes)
        if engine == "realesrgan":
            enhanced = self._enhance_with_realesrgan(image, preset)
            engine_label = "RealESRGAN_x4plus"
        else:
            enhanced = self._fallback_enhance(image, preset)
            engine_label = "Pillow fallback"

        output = self._encode(enhanced)
        return EnhancementResult(output, enhanced.width, enhanced.height, engine, engine_label, preset)

    def _decode(self, file_bytes: bytes) -> Image.Image:
        if not file_bytes:
            raise ValueError("Upload an image before enhancing.")
        try:
            image = Image.open(io.BytesIO(file_bytes))
            image = ImageOps.exif_transpose(image)
            image.load()
        except (UnidentifiedImageError, OSError) as exc:
            raise ValueError("The uploaded file is not a readable image.") from exc
        if image.width * image.height > self.max_input_pixels:
            raise ValueError("Image is too large. Use a file under 24 megapixels.")
        if image.mode not in ("RGB", "RGBA"):
            image = image.convert("RGB")
        if image.mode == "RGBA":
            background = Image.new("RGB", image.size, (255, 255, 255))
            background.paste(image, mask=image.getchannel("A"))
            image = background
        return image.convert("RGB")

    def _load_realesrgan(self):
        if self._realesrgan is not None:
            return self._realesrgan
        model = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=23, num_grow_ch=32, scale=4)
        self._realesrgan = RealESRGANer(
            scale=4,
            model_path=str(self.weights_path),
            model=model,
            tile=256 if self.device.type == "cuda" else 128,
            tile_pad=10,
            pre_pad=0,
            half=self.device.type == "cuda",
            device=self.device,
        )
        return self._realesrgan

    def _enhance_with_realesrgan(self, image: Image.Image, preset: PresetName) -> Image.Image:
        upsampler = self._load_realesrgan()
        rgb = np.array(image)
        bgr = rgb[:, :, ::-1]
        output_bgr, _ = upsampler.enhance(bgr, outscale=2)
        output_rgb = output_bgr[:, :, ::-1]
        enhanced = self._fit_long_edge(Image.fromarray(output_rgb), self.target_long_edge)
        return self._final_polish(enhanced, preset, denoise=False, strength="model")

    def _fallback_enhance(self, image: Image.Image, preset: PresetName) -> Image.Image:
        image = self._fit_long_edge(image, self.target_long_edge)
        return self._final_polish(image, preset, denoise=True, strength="fallback")

    def _final_polish(self, image: Image.Image, preset: PresetName, denoise: bool, strength: str) -> Image.Image:
        image = ImageOps.autocontrast(image, cutoff=1)
        if preset == "product_detail":
            color, contrast, sharpness = 1.04, 1.08, 1.26
            blur_radius, blend = 0.7, 0.28
        elif preset == "product_soft":
            color, contrast, sharpness = 1.01, 1.03, 1.08
            blur_radius, blend = 0.5, 0.18
        else:
            color, contrast, sharpness = 1.02, 1.05, 1.16
            blur_radius, blend = 0.6, 0.22
        if strength == "model":
            sharpness = max(1.02, sharpness - 0.06)
            blend *= 0.5
        if denoise:
            smoothed = image.filter(ImageFilter.MedianFilter(size=3))
            image = Image.blend(image, smoothed, blend)
        image = image.filter(ImageFilter.UnsharpMask(radius=blur_radius, percent=120, threshold=3))
        image = ImageEnhance.Contrast(image).enhance(contrast)
        image = ImageEnhance.Color(image).enhance(color)
        image = ImageEnhance.Sharpness(image).enhance(sharpness)
        return image

    def _fit_long_edge(self, image: Image.Image, long_edge: int) -> Image.Image:
        current_long_edge = max(image.size)
        if current_long_edge == long_edge:
            return image
        scale = long_edge / current_long_edge
        size = (max(1, round(image.width * scale)), max(1, round(image.height * scale)))
        return image.resize(size, Image.Resampling.LANCZOS)

    def _encode(self, image: Image.Image) -> bytes:
        buffer = io.BytesIO()
        image.save(buffer, format="JPEG", quality=94, optimize=True, progressive=True)
        return buffer.getvalue()

def data_url_from_bytes(image_bytes: bytes, content_type: str = "image/jpeg") -> str:
    return f"data:{content_type};base64,{base64.b64encode(image_bytes).decode('ascii')}"

def make_side_by_side(original: Image.Image, enhanced: Image.Image, panel_width: int = 480) -> Image.Image:
    def fit(image: Image.Image) -> Image.Image:
        image = ImageOps.exif_transpose(image).convert("RGB")
        height = max(1, round(panel_width * image.height / image.width))
        return image.resize((panel_width, height), Image.Resampling.LANCZOS)

    left = fit(original)
    right = fit(enhanced)
    label_height = 42
    gap = 20
    canvas = Image.new("RGB", (panel_width * 2 + gap, max(left.height, right.height) + label_height), "white")
    draw = ImageDraw.Draw(canvas)
    draw.text((0, 12), "Original", fill=(20, 23, 20))
    draw.text((panel_width + gap, 12), "Enhanced", fill=(20, 23, 20))
    canvas.paste(left, (0, label_height))
    canvas.paste(right, (panel_width + gap, label_height))
    return canvas

print("Device:", "GPU" if torch.cuda.is_available() else "CPU")
print("Pipeline ready.")

In [ ]:
# @title 4. Upload an inventory image
from google.colab import files
from IPython.display import display

uploaded = files.upload()
if not uploaded:
    raise RuntimeError("No file uploaded.")

input_name, input_bytes = next(iter(uploaded.items()))
original_image = Image.open(io.BytesIO(input_bytes))
print(f"Uploaded: {input_name} | {len(input_bytes) / 1024:.1f} KB | {original_image.width} x {original_image.height}")
display(original_image)

In [ ]:
# @title 5. Enhance and compare
engine = "realesrgan" # @param ["realesrgan", "pillow_fallback"]
preset = "product_standard" # @param ["product_standard", "product_detail", "product_soft"]
target_long_edge = 1800 # @param {type:"slider", min:800, max:2400, step:100}

enhancer = ProductImageEnhancer(WEIGHTS_PATH, target_long_edge=target_long_edge)
result = enhancer.enhance(input_bytes, preset=preset, engine=engine)

enhanced_image = Image.open(io.BytesIO(result.image_bytes))
output_path = Path("/content/enhanced_product.jpg")
output_path.write_bytes(result.image_bytes)

print(f"Engine: {result.engine_label}")
print(f"Output: {result.width} x {result.height}")
print(f"Saved to: {output_path}")

comparison = make_side_by_side(original_image, enhanced_image)
display(comparison)

In [ ]:
# @title 6. Create an API-style response and download the output
import json
from google.colab import files

api_response = {
    "image": data_url_from_bytes(result.image_bytes),
    "width": result.width,
    "height": result.height,
    "engine": result.engine,
    "engine_label": result.engine_label,
    "preset": result.preset,
}

response_path = Path("/content/enhancement_response.json")
response_path.write_text(json.dumps(api_response, indent=2))

print("API-style response written to:", response_path)
print("Enhanced image written to:", output_path)
files.download(str(output_path))